# 多租户操作
多租户提供数据隔离。每个租户存储在单独的分片上。存储在一个租户中的数据对其他租户不可见。如果您的应用程序服务于许多不同的用户，多租户可以保护他们的数据隐私，并提高数据库操作的效率。
 
> 租户状态已重命名v1.26     
> 在 中v1.26，HOT状态重命名为ACTIVE，COLD状态重命名为INACTIVE。

## 启用多租户
默认情况下，多租户功能处于禁用状态。要启用多租户功能，请multiTenancyConfig在集合定义中设置：

In [ ]:
client.schema.create_class({
    "class": "MultiTenancyCollection",
    "multiTenancyConfig": {"enabled": True}
})

In [ ]:
from weaviate.classes.config import Configure

multi_collection = client.collections.create(
    name="MultiTenancyCollection",
    # Enable multi-tenancy on the new collection
    multi_tenancy_config=Configure.multi_tenancy(enabled=True)
)

## 自动添加新租户

默认情况下，如果您尝试将对象插入不存在的租户，Weaviate 会返回错误。要更改此行为以便 Weaviate 创建新租户，请在集合定义中将其设置autoTenantCreation为。true

自动租户功能可用于v1.25.0批量导入，也可用于v1.25.2单个对象插入。

在创建集合时设置autoTenantCreation，或根据需要重新配置集合以更新设置。

导入大量对象时，自动创建租户非常有用。如果您的数据可能存在细微的不一致或拼写错误，请务必谨慎。例如，名称TenantOne、tenantOne和TenntOne会创建三个不同的租户。

In [ ]:
from weaviate.classes.config import Configure

multi_collection = client.collections.create(
    name="CollectionWithAutoMTEnabled",
    # Enable automatic tenant creation
    multi_tenancy_config=Configure.multi_tenancy(
        enabled=True,
        auto_tenant_creation=True
    )
)

### 创建集合


In [ ]:
from weaviate.classes.config import Configure

multi_collection = client.collections.create(
    name="CollectionWithAutoMTEnabled",
    # Enable automatic tenant creation
    multi_tenancy_config=Configure.multi_tenancy(
        enabled=True,
        auto_tenant_creation=True
    )
)

### 更新集合
使用客户端更新自动租户创建设置。自动租户仅适用于批量插入。

In [ ]:
from weaviate.classes.config import Reconfigure

collection = client.collections.get(collection_name)

collection.config.update(
    multi_tenancy_config=Reconfigure.multi_tenancy(auto_tenant_creation=True)
)

## 手动添加新租户
要将租户添加到集合，请指定集合和新租户。（可选）将租户活动状态指定为ACTIVE(可用、默认)、INACTIVE(不可用、在磁盘上) 或OFFLOADED(不可用、已卸载到云端)。



In [ ]:
from weaviate import Tenant

client.schema.add_class_tenants(
  class_name="MultiTenancyCollection",  # The class to which the tenants will be added
  tenants=[Tenant(name="tenantA"), Tenant(name="tenantB")]
)

In [ ]:
from weaviate.classes.tenants import Tenant

# Add two tenants to the collection
multi_collection.tenants.create(
    tenants=[
        Tenant(name="tenantA"),
        Tenant(name="tenantB"),
    ]
)

## 列出所有租户
列出集合中的现有租户。

此示例列出了集合中的租户MultiTenancyCollection：

In [ ]:
tenants = client.schema.get_class_tenants(
    class_name="MultiTenancyCollection"  # The class from which the tenants will be retrieved
)

In [ ]:
multi_collection = client.collections.get("MultiTenancyCollection")

tenants = multi_collection.tenants.get()

print(tenants)

## 按姓名获取租户

按名称从集合中获取租户。请注意，响应中会忽略不存在的租户名称。

此示例从集合中返回tenantA和：tenantBMultiTenancyCollection

In [ ]:
multi_collection = client.collections.get("MultiTenancyCollection")

tenant_names = ["tenantA", "tenantB", "nonExistentTenant"]  # `nonExistentTenant`` does not exist and will be ignored
tenants_response = multi_collection.tenants.get_by_names(tenant_names)

for k, v in tenants_response.items():
    print(k, v)

## 获得一名租户
从集合中获取特定租户。

此示例从集合中返回一个租户MultiTenancyCollection：

In [ ]:
multi_collection = client.collections.get("MultiTenancyCollection")

tenant_obj = multi_collection.tenants.get_by_name(tenant_name)

print(tenant_obj.name)

## 删除租户
要从集合中删除租户，请指定集合（例如MultiTenancyCollection）和租户（tenantB和tenantX）。如果指定的租户不属于集合，则删除操作将忽略租户名称。

In [ ]:
client.schema.remove_class_tenants(
    class_name="MultiTenancyCollection",  # The class from which the tenants will be removed
    tenants=["tenantB", "tenantX"]  # The tenants to be removed. tenantX will be ignored.
)

In [ ]:
multi_collection = client.collections.get("MultiTenancyCollection")

# Remove a list of tenants - tenantX will be ignored.
multi_collection.tenants.remove(["tenantB", "tenantX"])

## 管理租户状态
ACTIVE在、INACTIVE和之间更改租户状态OFFLOADED。

In [ ]:
from weaviate.classes.tenants import Tenant, TenantActivityStatus

multi_collection = client.collections.get("MultiTenancyCollection")
multi_collection.tenants.update(tenants=[
    Tenant(
        name="tenantA",
        activity_status=TenantActivityStatus.ACTIVE # INACTIVE, OFFLOADED
    )
])

## CRUD 操作
多租户集合要求tenantA每次 CRUD 操作都需要租户名称（例如），如下面的对象创建示例所示。

In [ ]:
object_id = client.data_object.create(
      class_name="MultiTenancyCollection",  # The class to which the object will be added
      data_object={
          "question": "This vector DB is OSS & supports automatic property type inference on import"
      },
      tenant="tenantA"  # The tenant to which the object will be added
)

In [ ]:
multi_collection = client.collections.get("MultiTenancyCollection")

# Get collection specific to the required tenant
multi_tenantA = multi_collection.with_tenant("tenantA")

# Insert an object to tenantA
object_id = multi_tenantA.data.insert(
    properties={
        "question": "This vector DB is OSS & supports automatic property type inference on import"
    }
)

## 搜索查询

多租户集合要求每个Get和Aggregate查询操作都包含租户名称（例如tenantA）。

In [ ]:
result = (
    client.query.get("MultiTenancyCollection", ["question"])
    .with_tenant("tenantA")
    .do()
)

In [ ]:
multi_collection = client.collections.get("MultiTenancyCollection")

# Get collection specific to the required tenant
multi_tenantA = multi_collection.with_tenant("tenantA")

# Query tenantA
result = multi_tenantA.query.fetch_objects(
    limit=2,
)

print(result.objects[0].properties)

## 交叉引用

可以从多租户集合对象添加交叉引用到：

- 非多租户集合对象，或
- 属于同一租户的对象。

tenantA多租户集合在创建、更新或删除交叉引用时需要租户名称（例如）。

In [ ]:
# Add the cross-reference property to the multi-tenancy class
client.schema.property.create("MultiTenancyCollection", {
    "name": "hasCategory",
    "dataType": ["JeopardyCategory"],
})

client.data_object.reference.add(
    from_uuid=object_id,  # MultiTenancyCollection object id (a Jeopardy question)
    from_class_name="MultiTenancyCollection",
    from_property_name="hasCategory",
    tenant="tenantA",
    to_class_name="JeopardyCategory",
    to_uuid=category_id
)

In [ ]:
from weaviate.classes.config import ReferenceProperty

multi_collection = client.collections.get("MultiTenancyCollection")
# Add the cross-reference property to the multi-tenancy class
multi_collection.config.add_reference(
    ReferenceProperty(
        name="hasCategory",
        target_collection="JeopardyCategory"
    )
)

# Get collection specific to the required tenant
multi_tenantA = multi_collection.with_tenant(tenant="tenantA")

# Add reference from MultiTenancyCollection object to a JeopardyCategory object
multi_tenantA.data.reference_add(
    from_uuid=object_id,  # MultiTenancyCollection object id (a Jeopardy question)
    from_property="hasCategory",
    to=category_id # JeopardyCategory id
)